The pipeline recursively scans heuristic-generation outputs under the project data directory (data/), automatically discovering all all_programs folders.

For each JSON file, it applies filtering rules (filename-level _Exception and JSON-level "exception": true), extracts relevant sub-fields, and constructs a cleaned dataset.

The results are aggregated into:

one global dataset containing all tasks, and

multiple per-task datasets split by normalized task name.

All outputs are saved in both .pkl and .csv formats under:

data/processed/ (global dataset),

data/processed/per_task/ (per-task datasets).

The pipeline is currently executed from the notebooks/ directory and assumes the project root contains a data/ folder with the raw heuristic outputs.




In [4]:
# -*- coding: utf-8 -*-
"""
Evolutionary Heuristic Design Data Pipeline.

This module scans heuristic-generation outputs, applies strict filtering
rules aligned with team consensus established through joint comparison and
review of data processing workflows by Guang-Yu (Dora) and Yunshi, and
produces clean, auditable datasets.

Contributors:
    Guang-Yu Yang, Yunshi Zou

Notes:
    - Code comments were generated with assistance from ChatGPT.
    - Final review and verification performed by Yunshi.

Key features:
1. Recursive auto-discovery via os.walk.
2. Dual-level exception filtering aligned through joint review:
   - Filename-level "_Exception".
   - JSON-level {"exception": true}.
3. Full audit trail with task-level traceability.
4. Global and per-task dataset persistence in both pickle and CSV formats.
"""

import json
import os
from typing import Dict
from typing import Union
from pathlib import Path
import pandas as pd


# =============================================================================
# Audit Statistics (Global)
# =============================================================================

audit_stats: Dict[str, int] = {
    "1_total_files_found": 0,
    "2_excluded_filename_exception": 0,
    "3_excluded_json_inner_exception": 0,
    "4_excluded_json_error": 0,
    "5_excluded_missing_offspring": 0,
    "6_dropped_nan_objective": 0,
    "7_dropped_empty_code": 0,
    "8_dropped_duplicate_id": 0,
    "FINAL_DATASET_COUNT": 0,
}

task_audit: Dict[str, Dict[str, int]] = {}


# =============================================================================
# Helper Functions
# =============================================================================

def fast_parse_strategy(filename: str) -> str:
    """Extract heuristic strategy identifier from filename."""
    parts = filename.split("_")
    try:
        if "op" in parts:
            idx = parts.index("op")
            return parts[idx + 1]
    except (ValueError, IndexError):
        pass
    return "unknown"


def update_task_stat(task_name: str, stat_key: str) -> None:
    """Increment audit counters for a specific task."""
    if task_name not in task_audit:
        task_audit[task_name] = {
            "raw": 0,
            "kept": 0,
            "diff_json_exception": 0,
        }
    task_audit[task_name][stat_key] += 1


def normalize_task_name(raw_app_type: str, instance_scale: str) -> str:
    """Normalize task naming across heterogeneous directory conventions."""
    name_map = {
        "bin_greedy": "BinPacking",
        "cvrp_lns": "CVRP",
        "premarshalling_astar": "Premarshalling",
        "puzzle_astar": "SlidingPuzzle",
    }

    base_name = name_map.get(raw_app_type, raw_app_type)

    if base_name in {"HotAI Material", ""}:
        for key, standard_name in name_map.items():
            if key in instance_scale:
                return standard_name
        return instance_scale

    return base_name


# =============================================================================
# Stage 1: Dataset Scanning
# =============================================================================

def scan_all_heuristic_datasets(
    root_path: Union[str, Path]
) -> pd.DataFrame:
    root_path = Path(root_path)
    all_data = []
    print(f"Starting directory scan at: {root_path}")

    for root, _, files in os.walk(root_path):
        if os.path.basename(root) != "all_programs":
            continue

        # root = .../<task>/<instance>/all_programs
        root_path_obj = Path(root)
        if root_path_obj.name != "all_programs":
            continue
        # instance directory = parent of all_programs
        instance_dir = root_path_obj.parent
    
        # task directory = first directory under DATA_ROOT
        try:
            task_dir = instance_dir
            while task_dir.parent != root_path:
                task_dir = task_dir.parent
        except Exception:
            continue
        raw_app_type = task_dir.name
        instance_scale = instance_dir.name
        current_task = normalize_task_name(raw_app_type, instance_scale)

        for filename in files:
            if not filename.endswith(".json"):
                continue

            audit_stats["1_total_files_found"] += 1
            update_task_stat(current_task, "raw")

            if "_Exception" in filename:
                audit_stats["2_excluded_filename_exception"] += 1
                continue

            file_path = os.path.join(root, filename)

            try:
                with open(file_path, "r", encoding="utf-8") as f:
                    content = json.load(f)

                if content.get("exception"):
                    audit_stats["3_excluded_json_inner_exception"] += 1
                    update_task_stat(current_task, "diff_json_exception")
                    continue

                offspring = content.get("offspring")
                if not isinstance(offspring, dict):
                    audit_stats["5_excluded_missing_offspring"] += 1
                    continue

                algorithm = offspring.get("algorithm", [""])
                code = offspring.get("code", [])
                objective = offspring.get("objective")

                heuristic_id = (
                    offspring.get("offspring_id")
                    or offspring.get("id")
                    or filename
                )

                all_data.append(
                    {
                        "heuristic_id": str(heuristic_id),
                        "raw_app_type": raw_app_type,
                        "instance_scale": instance_scale,
                        "filename": filename,
                        "strategy": fast_parse_strategy(filename),
                        "algorithm": (
                            algorithm[0]
                            if isinstance(algorithm, list)
                            else algorithm
                        ).strip("{} "),
                        "code": "".join(code)
                        if isinstance(code, list)
                        else code,
                        "objective": objective,
                    }
                )

            except (json.JSONDecodeError, OSError):
                audit_stats["4_excluded_json_error"] += 1
                continue

    return pd.DataFrame(all_data)


# =============================================================================
# Stage 2: Cleaning & Deduplication
# =============================================================================

def clean_and_dedup(df: pd.DataFrame) -> pd.DataFrame:
    """Clean dataset and remove invalid or duplicated entries."""
    if df.empty:
        return df

    df = df.copy()

    df["objective"] = pd.to_numeric(df["objective"], errors="coerce")
    before = len(df)
    df = df.dropna(subset=["objective"])
    audit_stats["6_dropped_nan_objective"] += before - len(df)

    before = len(df)
    df = df[df["code"].str.strip() != ""]
    audit_stats["7_dropped_empty_code"] += before - len(df)

    before = len(df)
    df = df.drop_duplicates(subset=["heuristic_id"], keep="first")
    audit_stats["8_dropped_duplicate_id"] += before - len(df)

    return df


def finalize_task_name(row: pd.Series) -> str:
    """Apply final task-name normalization."""
    return normalize_task_name(row["raw_app_type"], row["instance_scale"])


# =============================================================================
# Path Resolution (Notebook-vision)
# =============================================================================

# Current working directory: notebooks/XXXX
CURRENT_DIR = Path.cwd()

# Project root: ppp-performance-prediction
PROJECT_ROOT = CURRENT_DIR.parents[1]

# Data root directory
DATA_ROOT = PROJECT_ROOT / "data"

# Raw heuristic data directory (contains ./ all_programs / ...)
ROOT_MATERIAL = DATA_ROOT 

# Fail early if the expected data directory is missing
assert ROOT_MATERIAL.exists(), f"Data directory not found: {ROOT_MATERIAL}"

# =============================================================================
# Execution
# =============================================================================


df_raw = scan_all_heuristic_datasets(ROOT_MATERIAL)

if df_raw.empty:
    print("Error: No data found.")
    raise SystemExit(1)

df_final = clean_and_dedup(df_raw)
df_final["task_name"] = df_final.apply(finalize_task_name, axis=1)

# Timeout tagging
TIMEOUT_THRESHOLD = 10_000
df_final["is_timeout"] = False
df_final.loc[
    df_final["task_name"].str.contains("Puzzle|Premarshalling", case=False)
    & (df_final["objective"] > TIMEOUT_THRESHOLD),
    "is_timeout",
] = True

audit_stats["FINAL_DATASET_COUNT"] = len(df_final)

for task in df_final["task_name"].unique():
    if task in task_audit:
        task_audit[task]["kept"] = len(
            df_final[df_final["task_name"] == task]
        )

# =============================================================================
# Persistence
# =============================================================================

# Processed data root
OUTPUT_ROOT = DATA_ROOT / "processed"
PER_TASK_DIR = OUTPUT_ROOT / "per_task"

# Ensure output directories exist
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
PER_TASK_DIR.mkdir(parents=True, exist_ok=True)

# Global dataset
PICKLE_PATH = OUTPUT_ROOT / "all_heuristics_dataset.pkl"
CSV_PATH = OUTPUT_ROOT / "all_heuristics_dataset.csv"

df_final.to_pickle(PICKLE_PATH)
df_final.to_csv(CSV_PATH, index=False)

# Per-task datasets
for task_name, df_task in df_final.groupby("task_name"):
    safe_task_name = str(task_name).replace(" ", "_")
    task_pickle = PER_TASK_DIR / f"{safe_task_name}.pkl"
    task_csv = PER_TASK_DIR / f"{safe_task_name}.csv"
    df_task.to_pickle(task_pickle)
    df_task.to_csv(task_csv, index=False)


# =============================================================================
# Reporting
# =============================================================================

print("\n" + "=" * 70)
print(f"{'TASK NAME':<20} | {'RAW FILES':<10} | {'INTERNAL EXC':<12} | {'FINAL KEPT':<10}")
print("-" * 70)

for task, stats in task_audit.items():
    print(
        f"{task:<20} | "
        f"{stats['raw']:<10} | "
        f"{stats['diff_json_exception']:<12} | "
        f"{stats['kept']:<10}"
    )

print("-" * 70)
print("\n[Audit Breakdown]")
for k, v in audit_stats.items():
    print(f"{k:<30}: {v}")

print("=" * 70)
print(f"Global dataset saved to: {os.path.abspath(PICKLE_PATH)}")
print(f"Per-task datasets saved under: {os.path.abspath(PER_TASK_DIR)}")


Starting directory scan at: f:\KIT\HotAI\Task\ppp-performance-prediction\data

TASK NAME            | RAW FILES  | INTERNAL EXC | FINAL KEPT
----------------------------------------------------------------------
BinPacking           | 4847       | 0            | 4445      
CVRP                 | 1640       | 0            | 1557      
Premarshalling       | 4920       | 0            | 4647      
SlidingPuzzle        | 4920       | 0            | 4920      
----------------------------------------------------------------------

[Audit Breakdown]
1_total_files_found           : 16327
2_excluded_filename_exception : 758
3_excluded_json_inner_exception: 0
4_excluded_json_error         : 0
5_excluded_missing_offspring  : 0
6_dropped_nan_objective       : 0
7_dropped_empty_code          : 0
8_dropped_duplicate_id        : 0
FINAL_DATASET_COUNT           : 15569
Global dataset saved to: f:\KIT\HotAI\Task\ppp-performance-prediction\data\processed\all_heuristics_dataset.pkl
Per-task datasets sav